# Combine & Compare — all models

Reads whatever `results_all_models/<model>/<model>_results.csv` files exist on disk and builds the ranking + comparison plot. Run the per-model notebooks first (any subset, in any order); this works with however many have completed.

In [ ]:
# Minimal setup for combining per-model results (no dataset / training needed)
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT_RESULTS = "results_all_models"


In [ ]:
# ============================================================
# 16. Combine results from all COMPLETED models (reads CSVs on disk)
# ============================================================
# Rebuilds `combined` from whatever per-model result CSVs exist on disk, so the
# ranking + comparison plot below work even if you only ran some models today.

import glob as _glob

_result_csvs = sorted(_glob.glob(os.path.join(ROOT_RESULTS, "*", "*_results.csv")))
if not _result_csvs:
    raise FileNotFoundError(
        "No per-model results found yet. Run at least one model cell above first.")

all_dfs = [pd.read_csv(pth) for pth in _result_csvs]
combined = pd.concat(all_dfs, ignore_index=True)
combined.to_csv(os.path.join(ROOT_RESULTS, "ALL_experiments_combined.csv"), index=False)

print("Models included in combined results:", sorted(combined["model"].unique()))
print("Total experiments:", len(combined))
combined

In [ ]:
# ============================================================
# 16. Best experiment per model + ranking
# ============================================================

best_per_model = (
    combined
    .sort_values(["recall", "f1_score", "roc_auc"], ascending=False)
    .groupby("model", as_index=False)
    .first()
)

best_per_model = best_per_model.sort_values(
    ["recall", "f1_score", "roc_auc"], ascending=False
).reset_index(drop=True)
best_per_model.insert(0, "rank", best_per_model.index + 1)

ranking_cols = ["rank", "model", "experiment", "accuracy",
                "precision", "recall", "f1_score", "roc_auc"]
ranking_table = best_per_model[ranking_cols].copy()
ranking_table.to_csv(os.path.join(ROOT_RESULTS, "MODEL_RANKING.csv"), index=False)

print("FINAL MODEL RANKING (best experiment per model):")
ranking_table.round(4)

In [ ]:
# ============================================================
# 17. Visual comparison of models (best experiment each)
# ============================================================

plot_df = best_per_model.set_index("model")[
    ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
]
ax = plot_df.plot(kind="bar", figsize=(12, 6))
ax.set_title("Best experiment per model — metric comparison")
ax.set_ylabel("score")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.xticks(rotation=20, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(ROOT_RESULTS, "model_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()